In [17]:
import pandas as pd

In [18]:
df = pd.read_csv("/content/energy_clean.csv")

In [19]:
df["time"] = pd.to_datetime(df["time"], utc=True)       # ensures the time column is in datetime format before creating features

In [20]:
df = df.sort_values("time").reset_index(drop=True)     # keeps data in correct time order before making lag and rolling features.

In [21]:
print(df.head())

                       time     load  price
0 2014-12-31 23:00:00+00:00  25385.0  50.10
1 2015-01-01 00:00:00+00:00  24382.0  48.10
2 2015-01-01 01:00:00+00:00  22734.0  47.33
3 2015-01-01 02:00:00+00:00  21286.0  42.27
4 2015-01-01 03:00:00+00:00  20264.0  38.41


In [22]:
df["hour"] = df["time"].dt.hour        # extracts hour of day because electricity demand changes across 24 hours.

In [23]:
df["dayofweek"] = df["time"].dt.dayofweek       # extracts weekday number because weekday and weekend demand patterns are different

In [24]:
df["month"] = df["time"].dt.month           # extracts month to capture seasonal variation across the year.

In [25]:
df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)    # creates a weekend indicator because demand often changes on Saturday and Sunday.

In [26]:
print(df[["time", "hour", "dayofweek", "month", "is_weekend"]].head())      # checks whether the new time-based features were created correctly.

                       time  hour  dayofweek  month  is_weekend
0 2014-12-31 23:00:00+00:00    23          2     12           0
1 2015-01-01 00:00:00+00:00     0          3      1           0
2 2015-01-01 01:00:00+00:00     1          3      1           0
3 2015-01-01 02:00:00+00:00     2          3      1           0
4 2015-01-01 03:00:00+00:00     3          3      1           0


In [27]:
import numpy as np

In [28]:
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)      # converts hour into a cyclical sine feature to represent daily repeating pattern.

In [29]:
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)      # adds the cosine pair so the full daily cycle is represented properly.

In [30]:
df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)    # converts day of week into a cyclical feature for weekly repeating demand pattern.

In [31]:
df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)    # adds the cosine pair to complete the weekly cycle representation.

In [32]:
print(df[["hour", "hour_sin", "hour_cos", "dayofweek", "dow_sin", "dow_cos"]].head())

   hour  hour_sin  hour_cos  dayofweek   dow_sin   dow_cos
0    23 -0.258819  0.965926          2  0.974928 -0.222521
1     0  0.000000  1.000000          3  0.433884 -0.900969
2     1  0.258819  0.965926          3  0.433884 -0.900969
3     2  0.500000  0.866025          3  0.433884 -0.900969
4     3  0.707107  0.707107          3  0.433884 -0.900969


In [33]:
df["load_lag_1"] = df["load"].shift(1)       # uses the previous hour’s load to help predict the next hour.

In [34]:
df["load_lag_24"] = df["load"].shift(24)   # uses the same hour from the previous day, which is very useful in electricity demand forecasting.

In [35]:
df["load_lag_168"] = df["load"].shift(168)   # uses the same hour from the previous week to capture weekly repetition.

In [36]:
df["price_lag_1"] = df["price"].shift(1)     # uses the previous hour’s price as a predictor.

In [37]:
df["price_lag_24"] = df["price"].shift(24)    # uses the previous day’s same-hour price to capture daily price pattern

In [38]:
print(df[["load", "load_lag_1", "load_lag_24", "load_lag_168", "price", "price_lag_1", "price_lag_24"]].head(30))

       load  load_lag_1  load_lag_24  load_lag_168  price  price_lag_1  \
0   25385.0         NaN          NaN           NaN  50.10          NaN   
1   24382.0     25385.0          NaN           NaN  48.10        50.10   
2   22734.0     24382.0          NaN           NaN  47.33        48.10   
3   21286.0     22734.0          NaN           NaN  42.27        47.33   
4   20264.0     21286.0          NaN           NaN  38.41        42.27   
5   19905.0     20264.0          NaN           NaN  35.72        38.41   
6   20010.0     19905.0          NaN           NaN  35.13        35.72   
7   20377.0     20010.0          NaN           NaN  36.22        35.13   
8   20094.0     20377.0          NaN           NaN  32.40        36.22   
9   20637.0     20094.0          NaN           NaN  36.60        32.40   
10  22250.0     20637.0          NaN           NaN  43.10        36.60   
11  23547.0     22250.0          NaN           NaN  45.14        43.10   
12  24133.0     23547.0          NaN  

In [39]:
df["load_rollmean_24"] = df["load"].rolling(window=24).mean()        # creates the 24-hour average load to capture recent daily trend.

In [40]:
df["load_rollmean_168"] = df["load"].rolling(window=168).mean()      # creates the 168-hour average load to capture recent weekly trend.

In [41]:
df["price_rollmean_24"] = df["price"].rolling(window=24).mean()      # creates the 24-hour average price to capture recent price trend.

In [42]:
print(df[["load", "load_rollmean_24", "load_rollmean_168", "price", "price_rollmean_24"]].head(30))

       load  load_rollmean_24  load_rollmean_168  price  price_rollmean_24
0   25385.0               NaN                NaN  50.10                NaN
1   24382.0               NaN                NaN  48.10                NaN
2   22734.0               NaN                NaN  47.33                NaN
3   21286.0               NaN                NaN  42.27                NaN
4   20264.0               NaN                NaN  38.41                NaN
5   19905.0               NaN                NaN  35.72                NaN
6   20010.0               NaN                NaN  35.13                NaN
7   20377.0               NaN                NaN  36.22                NaN
8   20094.0               NaN                NaN  32.40                NaN
9   20637.0               NaN                NaN  36.60                NaN
10  22250.0               NaN                NaN  43.10                NaN
11  23547.0               NaN                NaN  45.14                NaN
12  24133.0              

In [43]:
df["y_load_tplus1"] = df["load"].shift(-1)         # creates the next-hour load as the prediction target for the model.

In [44]:
print(df[["time", "load", "y_load_tplus1"]].head(10))     # checks whether each row’s target is the next hour’s load.

                       time     load  y_load_tplus1
0 2014-12-31 23:00:00+00:00  25385.0        24382.0
1 2015-01-01 00:00:00+00:00  24382.0        22734.0
2 2015-01-01 01:00:00+00:00  22734.0        21286.0
3 2015-01-01 02:00:00+00:00  21286.0        20264.0
4 2015-01-01 03:00:00+00:00  20264.0        19905.0
5 2015-01-01 04:00:00+00:00  19905.0        20010.0
6 2015-01-01 05:00:00+00:00  20010.0        20377.0
7 2015-01-01 06:00:00+00:00  20377.0        20094.0
8 2015-01-01 07:00:00+00:00  20094.0        20637.0
9 2015-01-01 08:00:00+00:00  20637.0        22250.0


In [45]:
print(df[["time", "load", "y_load_tplus1"]].tail(5))  # checks the end of the dataset, where the last row should have a missing target.

                           time     load  y_load_tplus1
35023 2018-12-31 18:00:00+00:00  30653.0        29735.0
35024 2018-12-31 19:00:00+00:00  29735.0        28071.0
35025 2018-12-31 20:00:00+00:00  28071.0        25801.0
35026 2018-12-31 21:00:00+00:00  25801.0        24455.0
35027 2018-12-31 22:00:00+00:00  24455.0            NaN


In [46]:
print(df.isnull().sum())

time                   0
load                   0
price                  0
hour                   0
dayofweek              0
month                  0
is_weekend             0
hour_sin               0
hour_cos               0
dow_sin                0
dow_cos                0
load_lag_1             1
load_lag_24           24
load_lag_168         168
price_lag_1            1
price_lag_24          24
load_rollmean_24      23
load_rollmean_168    167
price_rollmean_24     23
y_load_tplus1          1
dtype: int64


In [47]:
df = df.dropna().reset_index(drop=True)

In [48]:
print(df.isnull().sum())

time                 0
load                 0
price                0
hour                 0
dayofweek            0
month                0
is_weekend           0
hour_sin             0
hour_cos             0
dow_sin              0
dow_cos              0
load_lag_1           0
load_lag_24          0
load_lag_168         0
price_lag_1          0
price_lag_24         0
load_rollmean_24     0
load_rollmean_168    0
price_rollmean_24    0
y_load_tplus1        0
dtype: int64


In [49]:
print(df.shape)

(34859, 20)


In [50]:
print(df.head())

                       time     load  price  hour  dayofweek  month  \
0 2015-01-08 05:00:00+00:00  22551.0  44.81     5          3      1   
1 2015-01-08 06:00:00+00:00  22543.0  44.81     6          3      1   
2 2015-01-08 07:00:00+00:00  24042.0  46.85     7          3      1   
3 2015-01-08 08:00:00+00:00  25932.0  49.00     8          3      1   
4 2015-01-08 09:00:00+00:00  27664.0  54.36     9          3      1   

   is_weekend  hour_sin      hour_cos   dow_sin   dow_cos  load_lag_1  \
0           0  0.965926  2.588190e-01  0.433884 -0.900969     22361.0   
1           0  1.000000  6.123234e-17  0.433884 -0.900969     22551.0   
2           0  0.965926 -2.588190e-01  0.433884 -0.900969     22543.0   
3           0  0.866025 -5.000000e-01  0.433884 -0.900969     24042.0   
4           0  0.707107 -7.071068e-01  0.433884 -0.900969     25932.0   

   load_lag_24  load_lag_168  price_lag_1  price_lag_24  load_rollmean_24  \
0      26992.0       25385.0        44.68         59.13  

In [51]:
df.to_csv("energy_features.csv", index=False)

In [52]:
print("energy_features.csv created successfully.")

energy_features.csv created successfully.


In [53]:
from google.colab import files
files.download("energy_features.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>